# TEST-104 OFFICIAL — chạm **LẦN THỨ HAI** · UniFormer-S ensemble 5 fold

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

⚠️⚠️ **Tập test này ĐÃ BỊ NHÌN MỘT LẦN** (2026-08-07, cấu hình cũ, macro-F1 0.6162).
Mọi báo cáo dùng số của lần chạy này **bắt buộc** nói rõ đây là lần chạm thứ hai.

Mọi lựa chọn đã khoá ở **`docs/TEST104_PREREGISTRATION.md` §B**, commit trước khi chạy.
**Không có tham số nào trong notebook này để sửa.** Sửa gì cũng là phá pre-registration,
và con số ra sẽ không dùng được để báo cáo.

## Cần mount gì

| | |
|---|---|
| **Cache lưới `128×128×16`** | sinh bởi `configs/preprocess_cghnet.yaml`, phải đủ **498** ca (394 trainval + 104 test) |
| **5 checkpoint** | sha256 đã ghim trong pre-registration §B1 |
| Internet | **KHÔNG cần** — không tải trọng số pretrained, checkpoint đã có sẵn |

⚠️ **Dataset checkpoint phải giữ nguyên cấu trúc thư mục `fold_N/`:**

```
<dataset>/fold_1/uniformer3D_best_1.pt
<dataset>/fold_2/uniformer3D_best_2.pt
...
```

Số trong tên file **và** số trong tên thư mục phải khớp nhau — hàm dò yêu cầu cả hai
nguồn cùng nói một fold. Upload phẳng 5 file vào một thư mục sẽ **không** được nhận, và
đó là cố ý: một thư mục gom lẫn nhiều run mà chỉ tin tên file thì sẽ ghép nhầm, tạo ra
"ensemble" đếm một model hai lần — với con số ra vẫn trông hoàn toàn hợp lý.

## Notebook này làm gì, và cố ý KHÔNG làm gì

Nó **chỉ suy luận và lưu xác suất**, không in một metric nào. Đọc số là việc của
`src.eval.test_report`, chạy trên CPU từ file `.npz` đã lưu.

Tách hai bước là có chủ đích: bước chạm test xảy ra **một lần** và tốn GPU. Gộp phần báo
cáo vào đây thì mỗi lần muốn xem thêm một metric lại phải chạy lại suy luận — mà "chạy
lại" trên test-104 chính là thứ bị cấm.

Ngoại lệ duy nhất được in ngay: **latency**. Nó chỉ đo được trong lúc chạy, và lần chạm
trước đã bỏ lỡ rồi không truy lại được.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- KHÔNG CÓ THAM SỐ NÀO ĐỂ SỬA -------------------------------------------
FOLDS = [1, 2, 3, 4, 5]
CONFIG_NAME = "uniformer_s.yaml"
PIN_SET = "uniformer"          # bộ sha256 đã ghim trong pre-registration §B1
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
# Clone ĐẦY ĐỦ, không --depth 1: cổng pre-registration kiểm bằng `git log` trên file đó,
# và một clone nông có thể không mang theo commit chứa nó.
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/test104_uniformer"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
PRE = load_yaml(REPO / "configs" / "preprocess_cghnet.yaml")
print(f"config: {CFG_PATH.name} · pin-set: {PIN_SET} · folds {FOLDS}")

## Cổng 0 ⚠️⚠️ — pre-registration §B đã commit chưa

Cổng này chạy **trước mọi thứ khác**. Một pre-registration viết sau khi nhìn số thì vô
nghĩa, nên nó được kiểm bằng `git log` chứ không bằng sự tồn tại của file.

`test_once` tự kiểm lại điều này lúc chạy; cell đây chỉ để bạn biết **ngay bây giờ** thay
vì sau khi đã mount xong mọi thứ.

In [ ]:
import subprocess as _sp

PREREG = "docs/TEST104_PREREGISTRATION.md"
_log = _sp.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short", "--", PREREG],
               cwd=REPO, capture_output=True, text=True).stdout.strip()
_dirty = _sp.run(["git", "status", "--porcelain", "--", PREREG],
                 cwd=REPO, capture_output=True, text=True).stdout.strip()

assert _log, f"⛔ {PREREG} CHƯA COMMIT — dừng. Commit và push trước khi chạy."
assert not _dirty, f"⛔ {PREREG} có thay đổi chưa commit: {_dirty!r}"

_noi_dung = (REPO / PREREG).read_text("utf-8")
assert "§B" in _noi_dung, "⛔ pre-registration chưa có mục §B cho lần chạm thứ hai"
for _khoa in ("uniformer_s.yaml", "62948396cdccd5a4", "8edf4fbc07f181b2"):
    assert _khoa in _noi_dung, f"⛔ §B không nhắc {_khoa} — sai bản pre-registration"

print("pre-registration commit:", _log)
print("cổng 0 ✓ — §B đã commit và sạch")

## 1. Cache và checkpoint

In [ ]:
import os as _os

INPUT_ROOT = Path("/kaggle/input")

# --- cache: tìm thư mục có cache_meta.json và đủ .npz ---
CACHE_DIR = None
for cand in sorted(INPUT_ROOT.glob("*/**/cache_meta.json")) + sorted(INPUT_ROOT.glob("*/cache_meta.json")):
    d = cand.parent
    if len(list(d.glob("*.npz"))) > 400:
        CACHE_DIR = d
        break
assert CACHE_DIR, (
    "⛔ không thấy cache đã mount. Cần Dataset chứa cache_meta.json + ~498 file .npz "
    "sinh bởi configs/preprocess_cghnet.yaml."
)
os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
print("cache:", CACHE_DIR)

# --- checkpoint: dùng chính hàm dò của test_once, không tự viết lại ---
from src.eval.test_once import find_checkpoints, sha16

CKPTS = {}
for cand in sorted({p.parent.parent for p in INPUT_ROOT.glob("*/**/*_best_*.pt")}
                   | {p.parent for p in INPUT_ROOT.glob("*/**/best_fold_*.pt")}
                   | set(INPUT_ROOT.glob("*"))):
    got = find_checkpoints(cand, FOLDS)
    if len(got) == len(FOLDS):
        CKPTS = got
        break
assert len(CKPTS) == len(FOLDS), (
    f"⛔ chỉ tìm được {sorted(CKPTS)} / {FOLDS}. Cần 5 file uniformer3D_best_N.pt "
    "nằm trong thư mục fold_N/ tương ứng."
)
for f in FOLDS:
    print(f"  fold {f}: {CKPTS[f]}")

## Cổng A ⚠️⚠️ — cache có ĐÚNG hình học mà model được train không

Cache sai hình học là chế độ hỏng **im lặng** tệ nhất ở đây: nếu lưới khác mà kích thước
vẫn hợp lệ, model chạy trơn, ra 104 xác suất trông hoàn toàn bình thường, và con số sai
sẽ được báo cáo như thật. Trên một tập chạm một lần thì không có cơ hội sửa.

Cổng đối chiếu `cache_meta.json` với **chính file config** trong repo, không với một bản
chép tay — hai bản chép sẽ trôi khỏi nhau.

In [ ]:
import json

meta = json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))

# Đọc kỳ vọng TỪ config trong repo, không ghi cứng ở đây.
CAN = {
    "axis_order": PRE["axis_order"],
    "align_phases": PRE["align_phases"],
    "reference_phase": PRE["reference_phase"],
    "crop_mode": PRE["crop_mode"],
    "target_spacing": PRE["target_spacing"],
    "target_size": PRE["target_size"],
    "crop_margin_voxels": PRE["crop_margin_voxels"],
}
for key, want in CAN.items():
    got = meta.get(key)
    assert got == want, f"⛔ cache SAI: {key} = {got!r}, cần {want!r}"
assert meta["lesion_tight"]["source"] == "mask", "⛔ phải cắt theo mask, không phải bbox"

# Model nhận đúng khối này; và crop_size của config train phải khớp target_size của cache.
assert list(CFG["data"]["crop_size"]) == list(PRE["target_size"]), (
    f"⛔ config train cắt {CFG['data']['crop_size']} nhưng cache dựng cho {PRE['target_size']}"
)

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, (
    f"⛔ chỉ có {n_npz} ca, cần 498. Cache thiếu test-104 — build_cache phải chạy trên "
    "toàn bộ annotation, không riêng trainval."
)
print(f"cổng A ✓ · hình học khớp config · {n_npz} ca · cache commit {meta.get('git_commit')}")

## Cổng B ⚠️⚠️ — checkpoint có đúng bộ đã ghim không

sha256 phải khớp danh sách trong pre-registration §B1. `test_once` cũng kiểm lại, nhưng
kiểm ở đây thì bạn biết **trước** khi cam kết lượt chạm.

Cổng cũng chặn hai checkpoint trùng nhau: một "ensemble" đếm cùng một model hai lần vẫn
cho ra con số trông hợp lý, nên không có dấu hiệu nào để phát hiện về sau.

In [ ]:
from src.eval.test_once import PIN_SETS

pinned = PIN_SETS[PIN_SET]
digests = {f: sha16(CKPTS[f]) for f in FOLDS}

print(f"{'fold':<6}{'sha256 đo được':<20}{'đã ghim':<20}")
ok = True
for f in FOLDS:
    khop = digests[f] == pinned.get(f)
    ok &= khop
    print(f"{f:<6}{digests[f]:<20}{pinned.get(f, '(thiếu)'):<20}{'✓' if khop else '⛔ LỆCH'}")

assert ok, "⛔ sha256 lệch bộ đã ghim — ĐÂY KHÔNG PHẢI bộ checkpoint đã khoá. DỪNG."
assert len(set(digests.values())) == len(FOLDS), "⛔ có checkpoint trùng nhau"
print("\ncổng B ✓ — đúng 5 checkpoint đã khoá, không cái nào trùng")

## 2. Chạm test-104

**Cell này là lần chạm.** Nó tự kiểm lại bốn thứ và **nổ** chứ không cảnh báo:

1. pre-registration đã commit (kiểm bằng `git log`)
2. `Splits.validate()` — trong đó có `val_fold_i ∩ test = ∅` với mọi `i`
3. sha256 khớp bộ đã ghim `uniformer`
4. không có hai checkpoint trùng nhau

104 ca × 5 model, khoảng một tới hai phút.

⚠️ **Không in metric ở đây, có chủ đích.** Xem phần đầu notebook.

In [ ]:
from src.eval.test_once import run as touch_test

CKPT_DIR = Path(_os.path.commonpath([str(p.parent.parent) for p in CKPTS.values()]))
print("checkpoint dir:", CKPT_DIR)
print("cache dir:     ", CACHE_DIR)
print("pin-set:       ", PIN_SET)

path = touch_test(
    ckpt_dir=CKPT_DIR,
    config_path=CFG_PATH,
    out_dir=os.environ["LLDMMRI_OUTPUT_DIR"],
    cache_dir=CACHE_DIR,
    folds=FOLDS,
    pin_set=PIN_SET,
)
print("\nđã lưu:", path)

## 3. Latency — con số DUY NHẤT được đọc ngay tại đây

Nó chỉ đo được trong lúc chạy. Lần chạm trước có sẵn con số này miễn phí, không ghi lại,
và sau đó không truy ra được nữa — test chạm một lần nên không chạy lại để đo được.

⚠️ **Đây là latency theo LÔ trên T4, không phải độ trễ một ca đơn lẻ.** Web app phục vụ
từng ca một nên sẽ chậm hơn con số này. Báo cáo phải nói rõ điều đó, đừng trình bày
`per_case` như thời gian đáp ứng của hệ thống thật.

In [ ]:
meta_run = json.loads((Path(path).parent / "test_run_meta.json").read_text("utf-8"))
lat = meta_run["latency"]

print(f"thiết bị        : {lat['device']} · batch {lat['batch_size']} · amp={lat['amp']}")
print(f"số ca           : {lat['n_cases']}")
print(f"giây mỗi model  : {lat['seconds_per_member']}")
print()
print(f"  1 model      : {lat['per_case_1model_ms']:>8.1f} ms/ca   (so được với văn liệu)")
print(f"  ensemble {len(lat['seconds_per_member'])}   : {lat['per_case_ensemble_ms']:>8.1f} ms/ca   "
      "(thứ hệ thống thật phải trả)")
print()
print("⚠ Đo theo lô. Một ca đơn lẻ sẽ chậm hơn — không có lợi thế theo lô, và cộng thêm")
print("  thời gian đọc + tiền xử lý NIfTI mà con số này KHÔNG bao gồm.")

## 4. Gói mang về

In [ ]:
import shutil

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
PACK = Path("/kaggle/working/test104_uniformer_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

for name in ("test_probs.npz", "test_run_meta.json"):
    src = OUT_ROOT / name
    assert src.exists(), f"⛔ thiếu {name}"
    shutil.copy2(src, PACK / name)

for f in sorted(PACK.rglob("*")):
    print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**10:.1f} KiB")

print()
print((PACK / "test_run_meta.json").read_text("utf-8"))

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz bản thân là zip; trình giải nén bung đệ quy sẽ
  biến nó thành thư mục và src.eval.* sẽ không thấy.

Đặt vào runs/test104_uniformer/ rồi chạy ở local (CPU, chạy lại bao nhiêu lần cũng được,
KHÔNG thành lần chạm thứ ba):

    python -m src.eval.test_report --run-dir runs/test104_uniformer \
        --oof-dir runs/Uniformer3D

  ⚠ --oof-dir BẮT BUỘC và phải là run out-of-fold của CHÍNH cấu hình này. `T` được fit ở
    đó rồi áp mù lên test. Trỏ nhầm sang run khác là fit T trên một phân bố xác suất khác
    — sai im lặng, không nổ.

Và phép so ghép cặp với lần chạm 1 trên đúng 104 ca đó — pre-registration §B6:

    python -m src.eval.compare --baseline runs/test104 --candidate runs/test104_uniformer
""")